# Caso 1 - Regresión: precio de casas (Housing)
Comparación de Regresión Lineal y Árbol de Decisión Regressor.

In [1]:
# Funciones auxiliares — Problema 1: Regresión (Housing)
# Son las mismas que usamos en los laboratorios de regresión lineal y árboles (caso MPG).
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error,
                             mean_squared_error, r2_score)


def plot_frequencies(df, columns, top_n=None):
    """Gráfica de barras con la frecuencia de cada categoría (top_n limita a las más comunes)."""
    for col in columns:
        freq = df[col].value_counts()
        if top_n:
            freq = freq.head(top_n)
        freq_df = freq.rename_axis(col).reset_index(name='Frecuencia')

        title = f'Frecuencias de {col}'
        if top_n and df[col].nunique() > top_n:
            title += f' (top {top_n})'

        fig = px.bar(freq_df, x=col, y='Frecuencia', title=title)
        fig.update_layout(xaxis={'categoryorder': 'total descending'})
        fig.show()


def evaluar_regresion(nombre, y_true, y_pred):
    """Imprime y regresa R², RMSE, MAE y MAPE (%) de un modelo de regresión."""
    metricas = {
        'modelo': nombre,
        'R²': round(r2_score(y_true, y_pred), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
        'MAE': round(mean_absolute_error(y_true, y_pred), 3),
        'MAPE (%)': round(mean_absolute_percentage_error(y_true, y_pred) * 100, 2),
    }
    print(f"{nombre}: R² {metricas['R²']} | RMSE {metricas['RMSE']} | "
          f"MAE {metricas['MAE']} | MAPE {metricas['MAPE (%)']}%")
    return metricas


def plot_importancias(modelo, columnas, top_n=None):
    """Importancia de cada variable en un modelo basado en árboles."""
    imp = pd.Series(modelo.feature_importances_, index=columnas).sort_values(ascending=True)
    if top_n:
        imp = imp.tail(top_n)

    fig = px.bar(
        x=imp.values, y=imp.index, orientation='h', text_auto='.3f',
        labels={'x': 'Importancia (reducción de impureza)', 'y': ''},
        title='Importancia de variables', template='plotly_white'
    )
    fig.update_layout(width=800, height=450)
    fig.show()
    return imp.sort_values(ascending=False)


In [2]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

df = pd.read_csv('https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/Housing.csv')
print(df.shape, 'nulos:', df.isnull().sum().sum())
print(df.dtypes)
print('Numéricas:', df.select_dtypes('number').shape[1], '| Categóricas:', df.select_dtypes(exclude='number').shape[1])

(545, 13) nulos: 0
price               int64
area                int64
bedrooms            int64
bathrooms           int64
stories             int64
mainroad              str
guestroom             str
basement              str
hotwaterheating       str
airconditioning       str
parking             int64
prefarea              str
furnishingstatus      str
dtype: object
Numéricas: 6 | Categóricas: 7


**Respuesta:** 6 numéricas (incluye `price`) y 7 categóricas.

In [3]:
binarias = ['mainroad','guestroom','basement','hotwaterheating','airconditioning','prefarea']
for c in binarias:
    df[c] = df[c].map({'yes': 1, 'no': 0})
df = pd.get_dummies(df, columns=['furnishingstatus'], drop_first=True, dtype=int)
X = df.drop(columns='price'); y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape); print(X_test.shape); print(y_train.shape); print(y_test.shape)

(436, 13)
(109, 13)
(436,)
(109,)


In [4]:
res = []
for nombre, m in [('Regresión Lineal', LinearRegression()), ('Árbol de Decisión', DecisionTreeRegressor(random_state=42))]:
    m.fit(X_train, y_train); p = m.predict(X_test)
    res.append(evaluar_regresion(nombre, y_test, p))
pd.DataFrame(res)

Regresión Lineal: R² 0.6529 | RMSE 1324506.9601 | MAE 970043.404 | MAPE 21.04%
Árbol de Decisión: R² 0.4771 | RMSE 1625669.9043 | MAE 1195266.055 | MAPE 24.93%


,modelo,R²,RMSE,MAE,MAPE (%)
0,Regresión Lineal,0.6529,1.324507e+06,970043.404,21.04
1,Árbol de Decisión,0.4771,1.625670e+06,1195266.055,24.93


**Elección: Regresión Lineal.** Tiene mayor R² (0.65 vs 0.48) y menor MAE y RMSE en test. El árbol sin límites memoriza el entrenamiento en un dataset pequeño (545 filas), y la regresión lineal es más interpretable.

**Fuga de información:** incluir `price` en un escalado previo al split. **Transformación inapropiada para binarias:** usar el valor como string.